# Change the Head of a classifier!

In this exercise we change the last layer of the NN to create a new classfiers.
The last laters of a deep NN are often reffered to as the head of the NN (i.e., the hed of the classifier)
Therefore, here we change the head od the classifier and retrain (only) it.
Lower layers extract high level features from the data, and are already trained
The head uses the features to classify

In [23]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import keras
os.environ["KERAS_BACKEND"] = "torch"

In [24]:
df = pd.read_csv('../dataset_big/fashion_mnist/fashion-mnist_train.csv')
x_train = df.drop('label', axis=1)
y_train = df['label']

df = pd.read_csv('../dataset_big/fashion_mnist/fashion-mnist_test.csv')
x_test = df.drop('label', axis=1)
y_test = df['label']

x_train.head(2)

,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [25]:
# Scale images to the [0, 1] range
x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255
print("x_train shape:", x_train.shape)
print("y_train shape:", y_train.shape)
print(x_train.shape[0], "train samples")
print(x_test.shape[0], "test samples")

x_train shape: (60000, 784)
y_train shape: (60000,)
60000 train samples
10000 test samples


In [26]:
x_train = x_train.values
y_train = y_train.values
x_train = x_train.reshape(x_train.shape[0], 28,28,1)

x_test = x_test.values
y_test = y_test.values
x_test = x_test.reshape(x_test.shape[0], 28,28,1)


print(x_train.shape)
print(y_train.shape)

(60000, 28, 28, 1)
(60000,)


In [27]:
input_shape = (28, 28, 1)

# Start from a pre-trained model

In [28]:
model = keras.saving.load_model("saved_fashion_mnistCONV.keras")
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 10, 10, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, 8, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 780,896 (2.98 MB)

 Trainable params: 260,298 (1016.79 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 520,598 (1.99 MB)

In [29]:
# I want to change the head of the model to output 2 classes only: "Ankle boot" vs "Not Ankle boot"
import tensorflow as tf
for layer in model.layers[:-1]:
    layer.trainable = False


new_model = keras.Sequential([keras.layers.Input(shape=input_shape)]+
                             model.layers[:-1])
new_model.add(keras.layers.Dense(2, activation='softmax'))



new_model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 26, 26, 64)     │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 24, 24, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 10, 10, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 8, 8, 128)      │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │           258 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 259,266 (1012.76 KB)

 Trainable params: 258 (1.01 KB)

 Non-trainable params: 259,008 (1011.75 KB)

 Ci sono  258 trainable parameters. Perchò il nodo precendete esce con 128 output, che diventano 128 input per i 2 nodi di output, con il termine noto, sono (128 + 1) * 2 = 258

In [30]:
# create new y_train and y_test
y_train_2 = np.where(y_train == 9, 1, 0)
y_test_2 = np.where(y_test == 9, 1, 0)


In [31]:
# train the new head
new_model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
history = new_model.fit(x_train, y_train_2, epochs=10, validation_split=0.2)

Epoch 1/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 13s 9ms/step - accuracy: 0.9602 - loss: 0.1084 - val_accuracy: 0.9923 - val_loss: 0.0252
Epoch 2/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9889 - loss: 0.0327 - val_accuracy: 0.9925 - val_loss: 0.0233
Epoch 3/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 20s 8ms/step - accuracy: 0.9902 - loss: 0.0279 - val_accuracy: 0.9928 - val_loss: 0.0210
Epoch 4/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9906 - loss: 0.0256 - val_accuracy: 0.9931 - val_loss: 0.0212
Epoch 5/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9911 - loss: 0.0243 - val_accuracy: 0.9928 - val_loss: 0.0206
Epoch 6/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9908 - loss: 0.0236 - val_accuracy: 0.9930 - val_loss: 0.0209
Epoch 7/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9909 - loss: 0.0238 - val_accuracy: 0.9933 - val_loss: 0.0204
Epoch 8/10
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 12s 8ms/step - accuracy: 0.9910 - loss: 0

In [32]:
# Predict the values from the test dataset
predictions = new_model.predict(x_test)
predictions = [np.argmax(p) for p in predictions]



313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step


In [33]:
# compute accuracy, confusion matrix, classification report
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
accuracy = accuracy_score(y_test_2, predictions)
conf_matrix = confusion_matrix(y_test_2, predictions)   
print("Accuracy:", accuracy)
print("Confusion Matrix:\n", conf_matrix)
print("Classification Report:\n", classification_report(y_test_2, predictions))


Accuracy: 0.9914
Confusion Matrix:
 [[8966   34]
 [  52  948]]
Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      1.00      9000
           1       0.97      0.95      0.96      1000

    accuracy                           0.99     10000
   macro avg       0.98      0.97      0.98     10000
weighted avg       0.99      0.99      0.99     10000

